# Comparação de Implementações de Gradient Descent

Este notebook demonstra e compara três implementações diferentes do algoritmo de Gradient Descent para regressão linear:

1. **Batch Gradient Descent** - Usa o conjunto completo de dados para cada atualização
2. **Stochastic Gradient Descent (SGD)** - Usa apenas uma instância aleatória para cada atualização
3. **Mini-batch Gradient Descent** - Usa um pequeno conjunto aleatório de instâncias para cada atualização

Vamos explorar como cada algoritmo se comporta, suas características de convergência e eficiência.

## Teoria: Gradient Descent e suas Variações

Gradient descent é uma técnica fundamental de otimização usada para minimizar funções, especialmente em modelos de machine learning. A ideia básica é atualizar iterativamente os parâmetros do modelo na direção oposta ao gradiente da função de custo.

### Principais Variações:

1. **Batch Gradient Descent**:
   - Usa o dataset inteiro para calcular o gradiente a cada iteração
   - Convergência mais estável e direta para o mínimo
   - Computacionalmente caro para grandes conjuntos de dados

2. **Stochastic Gradient Descent (SGD)**:
   - Usa apenas uma única instância aleatória para calcular o gradiente a cada iteração
   - Atualizações mais frequentes e caminhos ruidosos
   - Pode escapar de mínimos locais devido à sua natureza estocástica

3. **Mini-batch Gradient Descent**:
   - Meio termo entre as duas abordagens anteriores
   - Usa um pequeno conjunto aleatório de instâncias para cada atualização
   - Melhor equilíbrio entre estabilidade e eficiência computacional

Neste notebook, implementamos e comparamos essas três variações para entender suas características e comportamentos.

In [ ]:
# Importação das bibliotecas necessárias
import numpy as np
import matplotlib.pyplot as plt
import math, copy, time
import sys
from matplotlib.ticker import MaxNLocator

# Adiciona o diretório raiz do projeto ao sys.path para importar os módulos
import os
sys.path.append(os.path.abspath('..'))

# Importa os módulos do projeto
from src.utils.data_generator import generate_house_prices_data
from src.utils.cost_functions import compute_cost, compute_gradient, compute_model_output
from src.optimizers.gradient_descent import (
    batch_gradient_descent,
    stochastic_gradient_descent,
    minibatch_gradient_descent
)
from src.visualization.plotting import (
    plot_data_and_model,
    compare_cost_histories,
    compare_contour_plots
)

# Definição de cores para visualização
dlblue = '#0096ff'; dlorange = '#FF9300'; dldarkred='#C00000'; dlmagenta='#FF40FF'; dlpurple='#7030A0'
dlcolors = [dlblue, dlorange, dldarkred, dlmagenta, dlpurple]

## 1. Geração de Dados Sintéticos

Vamos gerar dados sintéticos para treinar nossos modelos de regressão linear. Usaremos um exemplo de preços de casas baseado no tamanho (em mil pés quadrados):

- O preço base de uma casa é $80.000
- Cada mil pés quadrados adiciona $120.000 ao preço
- Adicionamos um ruído gaussiano com desvio padrão de $20.000

In [ ]:
# Gerando os dados sintéticos
x_train, y_train, w_real, b_real = generate_house_prices_data(m=100, seed=42)

# Exibe os primeiros valores
print(f"x_train (primeiros 5): {x_train[:5]}")
print(f"y_train (primeiros 5): {y_train[:5]}")
print(f"Parâmetros reais: w={w_real}, b={b_real}\n")

# Visualiza os dados gerados
plt.figure(figsize=(10, 6))
plt.scatter(x_train, y_train, marker='x', c='r')
plt.title("Preços de Casas")
plt.xlabel("Tamanho (em mil pés²)")
plt.ylabel("Preço (em milhares)")
plt.grid(True)
plt.show()

## 2. Definição dos Hiperparâmetros

Para todos os algoritmos de gradient descent, vamos usar os mesmos hiperparâmetros iniciais para garantir uma comparação justa:

In [ ]:
# Hiperparâmetros
w_init = 0       # Valor inicial para o parâmetro w
b_init = 0       # Valor inicial para o parâmetro b
alpha = 1.0e-2   # Taxa de aprendizado
iterations = 10000  # Número de iterações
minibatch_size = 10  # Tamanho do mini-batch para Mini-batch GD

## 3. Batch Gradient Descent

Este algoritmo usa o conjunto completo de dados para calcular o gradiente em cada iteração. É a implementação mais estável, mas pode ser computacionalmente custosa para grandes conjuntos de dados.

In [ ]:
# Executa o Batch Gradient Descent
start_time = time.time()
w_bgd, b_bgd, J_hist_bgd, p_hist_bgd = batch_gradient_descent(
    x_train, y_train, w_init, b_init, alpha, iterations, compute_cost, compute_gradient)
bgd_time = time.time() - start_time

print(f"\nParâmetros encontrados por Batch Gradient Descent: w={w_bgd:.4f}, b={b_bgd:.4f}")
print(f"Diferença para os valores reais: w_diff={abs(w_bgd-w_real):.4f}, b_diff={abs(b_bgd-b_real):.4f}")
print(f"Tempo de execução: {bgd_time:.2f} segundos")

# Visualiza o resultado do modelo
plot_data_and_model(x_train, y_train, w_bgd, b_bgd, title="Batch Gradient Descent - Resultado")

## 4. Stochastic Gradient Descent (SGD)

Este algoritmo calcula o gradiente com base em apenas uma instância aleatória em cada iteração. É mais eficiente computacionalmente, mas tem um caminho de convergência mais ruidoso.

In [ ]:
# Executa o Stochastic Gradient Descent
start_time = time.time()
w_sgd, b_sgd, J_hist_sgd, p_hist_sgd = stochastic_gradient_descent(
    x_train, y_train, w_init, b_init, alpha, iterations, compute_cost, compute_gradient)
sgd_time = time.time() - start_time

print(f"\nParâmetros encontrados por SGD: w={w_sgd:.4f}, b={b_sgd:.4f}")
print(f"Diferença para os valores reais: w_diff={abs(w_sgd-w_real):.4f}, b_diff={abs(b_sgd-b_real):.4f}")
print(f"Tempo de execução: {sgd_time:.2f} segundos")

# Visualiza o resultado do modelo
plot_data_and_model(x_train, y_train, w_sgd, b_sgd, title="Stochastic Gradient Descent - Resultado")

## 5. Mini-batch Gradient Descent

Este algoritmo calcula o gradiente usando um pequeno conjunto aleatório de instâncias. É um compromisso entre o Batch GD e o SGD, equilibrando estabilidade e eficiência.

In [ ]:
# Executa o Mini-batch Gradient Descent
start_time = time.time()
w_mbgd, b_mbgd, J_hist_mbgd, p_hist_mbgd = minibatch_gradient_descent(
    x_train, y_train, w_init, b_init, alpha, iterations, compute_cost, compute_gradient, minibatch_size)
mbgd_time = time.time() - start_time

print(f"\nParâmetros encontrados por Mini-batch GD: w={w_mbgd:.4f}, b={b_mbgd:.4f}")
print(f"Diferença para os valores reais: w_diff={abs(w_mbgd-w_real):.4f}, b_diff={abs(b_mbgd-b_real):.4f}")
print(f"Tempo de execução: {mbgd_time:.2f} segundos")

# Visualiza o resultado do modelo
plot_data_and_model(x_train, y_train, w_mbgd, b_mbgd, title="Mini-batch Gradient Descent - Resultado")

## 6. Comparação dos Resultados

Agora, vamos comparar os resultados dos três algoritmos em termos de:
1. Parâmetros encontrados
2. Tempo de execução
3. Evolução do custo durante o treinamento
4. Caminhos de convergência no espaço de parâmetros

In [ ]:
# Tabela comparativa
from prettytable import PrettyTable

table = PrettyTable()
table.field_names = ["Método", "w", "b", "w_diff", "b_diff", "Tempo (s)"]
table.add_row(["Real", f"{w_real:.4f}", f"{b_real:.4f}", "N/A", "N/A", "N/A"])
table.add_row(["Batch GD", f"{w_bgd:.4f}", f"{b_bgd:.4f}", 
               f"{abs(w_bgd-w_real):.4f}", f"{abs(b_bgd-b_real):.4f}", f"{bgd_time:.2f}"])
table.add_row(["SGD", f"{w_sgd:.4f}", f"{b_sgd:.4f}", 
               f"{abs(w_sgd-w_real):.4f}", f"{abs(b_sgd-b_real):.4f}", f"{sgd_time:.2f}"])
table.add_row(["Mini-batch GD", f"{w_mbgd:.4f}", f"{b_mbgd:.4f}", 
              f"{abs(w_mbgd-w_real):.4f}", f"{abs(b_mbgd-b_real):.4f}", f"{mbgd_time:.2f}"])

print(table)

In [ ]:
# Comparação da evolução do custo
compare_cost_histories(
    [J_hist_bgd[::50], J_hist_sgd[::50], J_hist_mbgd[::50]],
    ["Batch GD", f"SGD (batch_size=1)", f"Mini-batch GD (batch_size={minibatch_size})"],
    title="Evolução do Custo durante o Treinamento"
)

In [ ]:
# Comparação dos caminhos no contorno
compare_contour_plots(
    x_train, y_train, 
    [p_hist_bgd, p_hist_sgd, p_hist_mbgd], 
    ["Batch GD", "SGD", f"Mini-batch (size={minibatch_size})"],
    compute_cost,
    title="Comparação dos Caminhos de Otimização"
)

## 7. Conclusões

### Comparação das Três Implementações de Gradient Descent

1. **Batch Gradient Descent**:
   - **Vantagens**: Mais estável e convergência direta para o mínimo global
   - **Desvantagens**: Mais custoso computacionalmente por iteração; requer mais memória para grandes datasets
   - **Observações**: Menos ruído na trajetória, como pode ser visto no gráfico de contorno

2. **Stochastic Gradient Descent (single instance)**:
   - **Vantagens**: Atualizações frequentes de parâmetros; computacionalmente mais leve por iteração
   - **Desvantagens**: Trajetória mais errática; pode levar mais tempo para convergir
   - **Observações**: Útil para datasets muito grandes onde uma passagem completa pelos dados é custosa

3. **Mini-batch Gradient Descent**:
   - **Vantagens**: Equilíbrio entre estabilidade e custo computacional; bom compromisso entre as duas abordagens anteriores
   - **Desvantagens**: Ainda apresenta ruído, embora menos que o SGD puro
   - **Observações**: É a abordagem mais comum em aplicações práticas de deep learning

### Recomendações Práticas

- **Datasets pequenos**: Use batch gradient descent para convergência mais estável
- **Datasets grandes**: Use mini-batches (tamanho entre 32 e 256 é comum na prática)
- **Otimização de hiperparâmetros**: Alpha (taxa de aprendizado) e tamanho do batch são críticos para o desempenho

O método mais adequado depende do problema específico, do tamanho do dataset e dos recursos computacionais disponíveis.